# Spatial Analysis

Where delays accumulate: top delay stops, district breakdown and line comparison.

## Setup

In [ ]:
from zh_tram_flow.notebook import *
import zh_tram_flow.analytics.spatial as an

TRAIN, TEST, lf = setup_analysis("03_analysis_4-spatial")
lf_all   = pl.concat([pl.scan_parquet(TRAIN), pl.scan_parquet(TEST)])
lf_delay = lf_all.filter(pl.col("canceled") == False)

%load_ext autoreload
%autoreload 2

## Top Delay Stops

Stops with highest average `arrival_delay` — potential bottleneck candidates.

In [ ]:
an.plot_top_delay_stops(lf_delay, cfg)

**Beobachtung:** Die höchsten Delay-Werte finden sich NICHT bei den zentralen Knotenpunkten, sondern bei **peripheren Endlinien-Haltestellen**.

**Top-10 Haltestellen nach Ø Delay:**
| Rang | Haltestelle | Ø Delay (s) | OTP | n |
|---:|:---|---:|---:|---:|
| 1 | Bertastrasse | **181.6** | 0.40 | 1'307 |
| 2 | Friedhof Sihlfeld | 167.0 | 0.50 | 1'307 |
| 3 | Friedrichstrasse | 144.6 | 0.60 | 14'375 |
| 4 | Frohburg | 116.7 | 0.70 | 14'390 |
| 5 | Albisgütli | 101.8 | 0.70 | 13'992 |
| 6 | Friedhof Enzenbühl | 93.8 | 0.70 | 292'204 |
| 7 | Balgrist | 85.2 | 0.80 | 292'940 |

**Interpretationshinweis:** Die Spitzenreiter 1–5 haben sehr niedrige Beobachtungszahlen (n=1'307 für Bertastrasse und Friedhof Sihlfeld) — das sind wahrscheinlich **Sonder- oder Eventlinien** (Albisgütli = L13/17 Sonderbetrieb; Frohburg/Friedhof Sihlfeld = Bestattungsfahrten?). Bei kleinem n sind extreme Mittelwerte statistisch instabil.

Ab Rang 6 (Friedhof Enzenbühl n=292'204) sind die Zahlen belastbar: Diese Haltestellen liegen auf **Aussenkorridoren** (Burgwies, Balgrist, Leutschenbach n=546'617) — nicht im Innenstadtkern. Das widerspricht der Annahme, dass zentrale Knotenpunkte die Delay-Hotspots seien.

**Frühankünfte (rechtes Chart):** Terminus-Haltestellen mit negativem Arrival Delay — Trams warten am Endpunkt auf den nächsten Abfahrtszeitpunkt. Das bestätigt das bimodale Delta-Muster aus `03_analysis_1-target`.

→ `bpuic` / `stop_name` als Feature; niedriges n als Qualitäts-Flag beachten; Sonderbetrieb-Haltestellen eventuell ausschliessen.

## Linien-Dichte vs. Verspätung

Wie viele verschiedene Linien bedienen jede Haltestelle — und sind die meistfrequentierten Knotenpunkte auch die verspätetsten?

In [ ]:
an.plot_lines_density_vs_delay(lf_delay, cfg)

In [ ]:
show_df(an.table_lines_density_vs_delay(lf_delay))

**Beobachtung:** **Kein Overlap zwischen den beiden Top-20-Listen** — exakt 0 Haltestellen erscheinen gleichzeitig in Top-20 nach Linienanzahl UND Top-20 nach Delay.

**Top-Haltestellen nach Linienanzahl (Knotenpunkte) — alle mit UNTERDURCHSCHNITTLICHEM Delay:**
| Haltestelle | Linien | Ø Delay (s) |
|:---|---:|---:|
| Haldenegg | 15 | **44.5** |
| Werd | 15 | 49.2 |
| Central | 15 | **48.3** |
| Stockerstrasse | 15 | 49.8 |
| Paradeplatz | 14 | **48.2** |
| Stauffacher | 15 | 60.7 |
| Bellevue | 14 | 55.0 |

**Kernbefund:** Die meistbediente Haltestelle Haldenegg (15 Linien) hat 44.5s Ø Delay — das liegt deutlich unter dem Netzschnitt (~56s). Paradeplatz (14 Linien, 48.2s), Central (15 Linien, 48.3s) — allesamt unter Durchschnitt. Einzig Stauffacher (15 Linien, 60.7s) liegt leicht über Durchschnitt.

**Konsequenz:** Die Hypothese "Mehr Linien = mehr Kaskadenrisiko = höherer Delay" findet in den Daten **keine Bestätigung** — sie ist widerlegt. Mögliche Erklärung: An den grossen Knotenpunkten ist der Betrieb besonders gut koordiniert (Fahrplan-Puffer, Fahrdienstleitung), während die echten Delay-Akkumulatoren auf den Aussenkorridoren liegen (→ F-SPAT-01 zu korrigieren).

→ Linienanzahl als Feature wenig vielversprechend für Delay-Prognose; besser: absolute Haltestelleneigenschaften (Korridor, Aussenlage) nutzen.

## Starthaltestellen-Diagnose

Starthaltestellen verzerren die Statistik: Trams warten am Startpunkt auf ihren Abfahrtszeitpunkt und "ankommen" weit vor dem Fahrplan — obwohl das kein echtes Betriebsproblem ist. Das zieht den Ø `arrival_delay` künstlich nach unten und beschönigt die Netz-Performance.

**Identifikation ohne stop_sequence:** Proxy-Kriterium — Starthaltestellen haben:
1. Sehr negatives `arrival_delay` (Tram steht schon lange da)
2. Positives `delay_delta` (Tram wartet, dann Abfahrt nahe Fahrplan → delta = dep_delay − arr_delay ist stark positiv)

In [ ]:
an.plot_start_stop_diagnosis(lf_delay, cfg)
show_df(an.table_start_stop_candidates(lf_delay))

**Beobachtung:** Die Starthaltestellen-Diagnose findet mit den gewählten Schwellenwerten (avg_arr < −30s UND avg_delta > +20s) **0 Kandidaten**. Die Verzerrung des Netzschnitts durch klassische Starthaltestellen beträgt **0.0s** — kein Bereinigungsbedarf mit dieser Methode.

**Interpretation:** Das Proxy-Kriterium "stark negatives Arrival + stark positives Delta" trifft nicht zu:
- Terminus-Haltestellen scheinen in den Daten entweder nicht mit stark negativem Arrival aufzutauchen (Trams werden erst kurz vor Abfahrt erfasst), oder der Fahrplan ist so gestrickt, dass arrival_delay dort nicht systematisch negativ ist.
- Die frühen Ankünfte aus dem Top-Delay-Chart (z.B. negative Bars rechts) sind vorhanden, erfüllen aber nicht die kombinierte Bedingung (fehlendes starkes Delta).

**Alternative Interpretation:** Die Top-Delay-Ausreisser (Bertastrasse 181.6s n=1'307, Friedhof Sihlfeld 167.0s n=1'307) sind keine klassischen Starthaltestellen, sondern Sonder-/Eventhalte — hoher Delay, aber keine Frühankunft. Diese können durch einen n-Mindest-Filter oder einen spezifischen Halt-Namen-Filter herausgefiltert werden, statt durch das Start-Stop-Proxy.

→ Starthaltestellen-Flag nicht als Feature sinnvoll (keine Kandidaten gefunden). Stattdessen `n_threshold` Filter für Low-Volume-Haltestellen in Modellierung einbauen.

## District Analysis

Average delay per Zurich district (Kreis 1–12 + outside). Identifies spatial delay clusters.

In [ ]:
an.plot_district_analysis(lf_delay, cfg)
show_df(an.table_district_analysis(lf_delay))

**Beobachtung:** Die Stadtkreis-Analyse zeigt ein klares Muster: **Aussenkreise haben höhere Verspätung als Innenstadt**.

**Ø Delay nach Stadtkreis (sortiert):**
| Stadtkreis | Ø Delay (s) | OTP |
|:---|---:|---:|
| **Kreis 11** | **68.3** | 83% |
| Kreis 12 | 66.3 | 85% |
| Kreis 8 | 63.7 | 85% |
| Kreis 9 | 59.7 | 87% |
| Kreis 7 | 58.7 | 87% |
| outside | 58.4 | 87% |
| ... | | |
| Kreis 1 | 51.3 | 88% |
| Kreis 10 | 51.0 | 88% |
| **Kreis 5** | **49.9** | **89%** |

**Kernbefund:** Kreis 11 (68.3s, OTP 83%) und Kreis 12 (66.3s) sind die problematischsten Kreise. Kreis 1 (Altstadt/Innenstadt) liegt mit 51.3s weit unten — trotz hohem Fahrgastaufkommen. Kreis 5 (Industriequartier) ist der pünktlichste Kreis (49.9s, OTP 89%).

**Interpretation:** Die peripheren Korridore (Kreise 11, 12 = Oerlikon/Schwamendingen-Achse, Seefeld/Weinegg) akkumulieren mehr Delay als die Innenstadtachsen. Das passt zum Befund aus F-SPAT-01 (Delay-Hotspots auf Aussenkorridoren). Kreis 8 (63.7s) überrascht — liegt an der Seefeld/Balgrist-Strecke (L2, L4) mit langen Fahrzeiten.

→ `district_nr` als Feature additiv zu `line_name` nützlich; Kreise 11/12 als High-Risk-Marker.

## Line Analysis

Delay profile per tram line — which lines are most unreliable?

In [ ]:
an.plot_line_analysis(lf_delay, cfg)
show_df(an.table_line_analysis(lf_delay))

**Beobachtung:** Die Linienanalyse zeigt erhebliche Unterschiede — Linie E (128s) als klarer Ausreisser, L11 (68.7s) mit Abstand schlechteste "reguläre" Linie.

**Linien-Ranking (Ø Arrival Delay):**
| Linie | Ø Delay (s) | OTP | Ø Delta (s) |
|:---|---:|---:|---:|
| E | 130.2 | 56% | −0.5 |
| **L11** | **68.7** | 82% | +6.2 |
| L15 | 61.4 | 85% | +2.0 |
| L10 | 60.1 | 85% | +6.5 |
| L8 | 59.7 | 85% | +4.5 |
| L6 | 38.4 | 93% | +4.2 |
| L51 | 41.4 | 93% | +20.2 |

**Delay-Delta:** Alle Linien haben **positives** Delta (Abfahrtsverspätung > Ankunftsverspätung) — Trams akkumulieren Delay an den Haltestellen, keine Linie baut Verspätung systematisch ab. L51 hat das grösste Delta (+20.2s) trotz niedrigstem Delay — kurze Linie mit vielen Warte-Momenten. L11 (+6.2s) und L10 (+6.5s) sind die stärksten Akkumulatoren unter den langen Hauptlinien.

**Linie E** (OTP 56%, Ø 130s): Sonderlinie/Entlastungslinie — bestätigt F-TARGET-12 und F-NET-08, kein Datenfehler.

→ `line_name` ist stärkster räumlicher Prädiktor. L11 als Hochrisiko-Linie für Modellpriorisierung.

## Feature: `dwell_time`

Geplante Haltezeit = `departure_schedule − arrival_schedule` in Sekunden (F-TARGET-04). Kurze Dwell-Time = wenig Puffer → höheres Verspätungsrisiko (F-TARGET-03). Zeigt welche Haltestellen und Linien strukturell zu wenig Zeit einplanen.

In [ ]:
an.plot_dwell_time(lf_delay, cfg)

In [ ]:
show_df(an.table_dwell_time_by_line(lf_delay))

**Beobachtung:** Die `dwell_time`-Verteilung zeigt ein überraschendes Ergebnis: **71.3% aller Halte haben dwell_time = 0s** — identisch mit dem ≤20s-Anteil.

**Ø dwell_time netzweit: 17.6s | Anteil 0s: 71.3%**

Das bedeutet: **Mehr als 2 von 3 Halten sind fahrplanmässig als Durchfahrten ohne geplante Haltezeit kodiert.** Der Median ist für alle Linien **0s**.

**Konsequenz für Feature-Nutzung:** Ein Scatter `dwell_time × delay` kann keinen negativen Zusammenhang zeigen, wenn 71% der Datenpunkte bei 0s liegen — die Streuung fehlt. `dwell_time` als kontinuierliches Feature ist damit nur für die 29% der Halte mit geplanter Haltezeit informativ. Als binäres Feature (`has_dwell = dwell_time > 0`) möglicherweise nützlicher.

**Linien-Unterschiede:** Die Ø-Werte variieren zwischen L51 (12.1s) und Linie E (24.2s) — aber alle haben Median=0. L3 hat mit 21.9s die höchste mittlere Haltezeit unter den regulären Hauptlinien, L10 die niedrigste (15.5s). Diese Unterschiede sind durch den 0-Anteil stark gedämpft.

→ `dwell_time` als Feature weniger stark als erhofft. Stattdessen `has_dwell` (binary) prüfen. Die vorliegende Modellierung sollte beide Varianten testen (F-SPAT-08 neu).

## Key Findings

→ Vollständige Findings-Tabelle mit Impact und Action in [`03_analysis_0-overview.ipynb`](03_analysis_0-overview.ipynb).

| ID | Finding | Status |
|:---|:---|:---|
| F-SPAT-01 | Delay-Hotspots sind **nicht** die zentralen Knotenpunkte, sondern periphere Aussenkorridore: Bertastrasse 181.6s, Friedhof Enzenbühl 93.8s, Balgrist 85.2s, Leutschenbach 82.7s. Top-2 haben n=1'307 (Sonder-/Eventlinien) — statistisch instabil. | done |
| F-SPAT-02 | Terminus-Haltestellen zeigen negative Delay-Werte (Frühankünfte) — Pufferzeit eingebaut; bestätigt bimodalen Delta-Cluster aus F-TARGET | done |
| F-SPAT-03 | Stadtkreis-Delays: Kreis 11 schlechtester (68.3s, OTP 83%), Kreis 12 (66.3s), Kreis 8 (63.7s). Innenstadt Kreis 1=51.3s (gut). Kreis 5 bester (49.9s, OTP 89%). | done |
| F-SPAT-04 | Alle Linien haben positives Delay-Delta — keine Linie baut Verspätung systematisch ab. Stärkste Akkumulatoren: L10 (+6.5s), L11 (+6.2s). L51 hat grösstes Delta (+20.2s) bei niedrigstem Delay. | done |
| F-SPAT-05 | `line_name` ist stärkster räumlicher Prädiktor; `district_nr` additiv nützlich. L11 (68.7s, OTP 82%) ist die kritischste Hauptlinie. | done |
| F-SPAT-06 | Starthaltestellen-Proxy (avg_arr < −30s AND avg_delta > +20s) findet 0 Kandidaten — keine Verzerrung nachweisbar. Stattdessen: n-Threshold-Filter für Low-Volume-Haltestellen empfohlen. | done |
| F-SPAT-07 | **Keine Korrelation** Linienanzahl × Delay: 0 Overlap zwischen Top-20 nach Linien und Top-20 nach Delay. Haldenegg (15 Linien, 44.5s), Paradeplatz (14 Linien, 48.2s) — alle unter Netzschnitt. Kaskadenrisiko-Hypothese widerlegt. | done |
| F-SPAT-08 | `dwell_time` = 0s für 71.3% aller Halte (Median=0 für alle Linien) — kein klarer Zusammenhang mit Delay messbar. Binäres Feature `has_dwell` als Alternative prüfen. | done |